In [1]:
# Clone the official open-source PKINN codebase
!git clone https://github.com/addm9/Photovoltaic_Knowledge_Informed_Neural_Network.git
%cd Photovoltaic_Knowledge_Informed_Neural_Network
!ls -la

Cloning into 'Photovoltaic_Knowledge_Informed_Neural_Network'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 31 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 6.43 MiB | 4.96 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Photovoltaic_Knowledge_Informed_Neural_Network
total 24
drwxr-xr-x 5 root root 4096 Aug  2 11:06 .
drwxr-xr-x 1 root root 4096 Aug  2 11:06 ..
drwxr-xr-x 2 root root 4096 Aug  2 11:06 code
drwxr-xr-x 2 root root 4096 Aug  2 11:06 Dataset
drwxr-xr-x 8 root root 4096 Aug  2 11:06 .git
-rw-r--r-- 1 root root 2381 Aug  2 11:06 README.md


In [2]:
# Check GPU availability and install dependencies
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Install any missing specific requirements if the repo includes a requirements.txt
!pip install -r requirements.txt

Num GPUs Available:  1
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
import os
os.chdir('/content/Photovoltaic_Knowledge_Informed_Neural_Network')
!pwd

/content/Photovoltaic_Knowledge_Informed_Neural_Network


In [4]:
!ls -la code/

total 48
drwxr-xr-x 2 root root  4096 Aug  2 11:06  .
drwxr-xr-x 5 root root  4096 Aug  2 11:06  ..
-rw-r--r-- 1 root root  9264 Aug  2 11:06 'Fluctuation intensity.py'
-rw-r--r-- 1 root root  8887 Aug  2 11:06  Physical-model.py
-rw-r--r-- 1 root root 16263 Aug  2 11:06  PKINN-Ensemble-model.py


In [6]:
# Path to script
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read script content
with open(script_path, 'r') as f:
    code = f.read()

# Replace placeholder with actual dataset path
updated_code = code.replace('"xxx.csv"', '"Dataset/xSi12922.csv"')
updated_code = updated_code.replace("'xxx.csv'", "'Dataset/xSi12922.csv'")

# Write updated script back
with open(script_path, 'w') as f:
    f.write(updated_code)

print("Path successfully updated!")

Path successfully updated!


In [8]:
import pandas as pd
df = pd.read_csv('/content/Photovoltaic_Knowledge_Informed_Neural_Network/Dataset/xSi12922.csv')
print("Columns in dataset:")
print(df.columns.tolist()[:15])  # Prints first 15 column names

Columns in dataset:
['date', 'POA', 'POA irradiance CMP22 pyranometer (W/m2)', 'PV module back surface temperature (degC)', 'Isc (A)', 'Pmp (W)', 'Imp (A)', 'Vmp (V)', 'Voc (V)', 'FF (%FF)', 'Change in CMP22 during I-V scan +/- W/m2', 'Change in Li-COR during I-V scan +/- W/m2', 'MT5 cabinet temperature (degC)', 'Dry bulb temperature (degC)', 'Relative humidity (%RH)']


In [11]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read existing script
with open(script_path, 'r') as f:
    lines = f.readlines()

# Find where data loading starts (around line 230-235)
cutoff_idx = 0
for i, line in enumerate(lines):
    if 'pd.read_csv' in line:
        cutoff_idx = i
        break

# Keep all function and model definitions before data loading
clean_lines = lines[:cutoff_idx]

# Append the fixed data loading and pre-processing block
fixed_tail = '''
# --- CLEAN DATA LOADING & FEATURE MAPPING ---
data_path = "Dataset/xSi12922.csv"
data = pd.read_csv(data_path)

# Parse date column into time components
if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"])
    data["Year"] = data["date"].dt.year
    data["Month"] = data["date"].dt.month
    data["Day"] = data["date"].dt.day
    data["Hour"] = data["date"].dt.hour
    data["Minute"] = data["date"].dt.minute

# Select features available in the NREL dataset
feature_cols = [
    "Year", "Month", "Day", "Hour", "Minute",
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "Dry bulb temperature (degC)",
    "Relative humidity (%RH)"
]

# Map available features safely
available_cols = [c for c in feature_cols if c in data.columns]
data_weather = data.loc[:, available_cols].copy()

print("Data loaded successfully!")
print("Selected features shape:", data_weather.shape)
'''

# Write the patched script back
with open(script_path, 'w') as f:
    f.writelines(clean_lines)
    f.write(fixed_tail)

print("PKINN-Ensemble-model.py successfully rewritten!")

PKINN-Ensemble-model.py successfully rewritten!


In [12]:
%cd /content/Photovoltaic_Knowledge_Informed_Neural_Network
!python code/PKINN-Ensemble-model.py


/content/Photovoltaic_Knowledge_Informed_Neural_Network
cuda
Data loaded successfully!
Selected features shape: (35861, 9)


In [13]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read existing script
with open(script_path, 'r') as f:
    code = f.read()

# Append full PKINN training, physical loss, and evaluation pipeline
training_pipeline = '''

import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Target variable extraction (Pmp / Power)
if 'Pmp (W)' in data.columns:
    target_power = data['Pmp (W)'].values
elif 'Pmp' in data.columns:
    target_power = data['Pmp'].values
else:
    target_power = data.iloc[:, 5].values  # Fallback to power column index

# 2. Scale Features & Target
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_x.fit_transform(data_weather.fillna(0))
y_scaled = scaler_y.fit_transform(target_power.reshape(-1, 1))

# 3. Train / Test Split (80% Train, 20% Test)
train_size = int(len(X_scaled) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y_scaled[:train_size], y_scaled[train_size:]

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).cuda() if torch.cuda.is_available() else torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).cuda() if torch.cuda.is_available() else torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32).cuda() if torch.cuda.is_available() else torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).cuda() if torch.cuda.is_available() else torch.tensor(y_test, dtype=torch.float32)

print(f"Dataset split complete: {len(X_train)} Train samples, {len(X_test)} Test samples")

# 4. Define Baseline Neural Model Architecture
class PKINNSimple(nn.Module):
    def __init__(self, input_dim):
        super(PKINNSimple, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.relu(self.fc2(out))
        out = self.fc3(out)
        return out

model = PKINNSimple(X_train.shape[1])
if torch.cuda.is_available():
    model = model.cuda()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 5. Model Training Loop
print("\\n--- Starting PKINN Baseline Model Training ---")
epochs = 100
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_t)
    loss = criterion(predictions, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] - Loss: {loss.item():.6f}")

# 6. Evaluation on Unseen Test Set
model.eval()
with torch.no_grad():
    test_preds_scaled = model(X_test_t).cpu().numpy()

test_preds = scaler_y.inverse_transform(test_preds_scaled)
y_test_actual = scaler_y.inverse_transform(y_test.reshape(-1, 1))

# 7. Metrics Calculation
rmse = np.sqrt(mean_squared_error(y_test_actual, test_preds))
mae = mean_absolute_error(y_test_actual, test_preds)
r2 = r2_score(y_test_actual, test_preds)

print("\\n==========================================")
print("     PKINN REPRODUCED BASELINE RESULTS     ")
print("==========================================")
print(f" Root Mean Squared Error (RMSE) : {rmse:.4f} W")
print(f" Mean Absolute Error (MAE)      : {mae:.4f} W")
print(f" Coefficient of Determination (R2): {r2:.4f}")
print("==========================================\\n")
'''

with open(script_path, 'a') as f:
    f.write(training_pipeline)

print("Full training pipeline successfully attached!")

Full training pipeline successfully attached!


In [14]:
!python code/PKINN-Ensemble-model.py

cuda
Data loaded successfully!
Selected features shape: (35861, 9)
Dataset split complete: 28688 Train samples, 7173 Test samples

--- Starting PKINN Baseline Model Training ---
Epoch [1/100] - Loss: 0.102954
Epoch [20/100] - Loss: 0.042988
Epoch [40/100] - Loss: 0.016028
Epoch [60/100] - Loss: 0.003049
Epoch [80/100] - Loss: 0.001244
Epoch [100/100] - Loss: 0.000871

     PKINN REPRODUCED BASELINE RESULTS     
 Root Mean Squared Error (RMSE) : 20.6228 W
 Mean Absolute Error (MAE)      : 16.8415 W
 Coefficient of Determination (R2): 0.3379



In [15]:
with open('/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py', 'r') as f:
    lines = f.readlines()

# Print lines 150 to 230 to see the authors' original training/loss functions
for i, line in enumerate(lines[150:230], start=151):
    print(f"{i}: {line}", end='')

151:         x = self.relu(self.fc1(torch.cat((x.float(), Isc.unsqueeze(1).float(), x_phy.unsqueeze(1).float(),I_MPP.unsqueeze(1).float(),P_MPP.unsqueeze(1).float()), dim=1)))
152:         x = self.relu(self.fc2(x))
153:         return x
154: 
155: 
156: # Define the shared part of the model
157: class SharedNet(nn.Module):
158:     def __init__(self, hidden_size, shared_hidden_sizes, output_size):
159:         super(SharedNet, self).__init__()
160:         self.fc3 = nn.Linear(hidden_size, shared_hidden_sizes[0])
161:         self.fc4 = nn.Linear(shared_hidden_sizes[0], shared_hidden_sizes[1])
162:         self.fc5 = nn.Linear(shared_hidden_sizes[1], output_size)
163:         self.relu = nn.ReLU()
164:         self.sigmoid = nn.ReLU()
165: 
166:     def forward(self, x):
167:         x = self.relu(self.fc3(x))
168:         x = self.relu(self.fc4(x))
169:         x = self.fc5(x)
170:         return x
171: 
172: 
173: # Combine all parts into one model
174: class CombinedNet(nn.Module):

In [16]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read original script
with open(script_path, 'r') as f:
    lines = f.readlines()

# Keep everything up to parameter definitions (around line 230)
cutoff = 0
for i, line in enumerate(lines):
    if 'Rs = 0.040091967' in line:
        cutoff = i + 5
        break

clean_head = lines[:cutoff]

# Append native data mapping + sequence generation + CombinedNet execution
native_execution_block = '''

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load Data
data_path = "Dataset/xSi12922.csv"
df = pd.read_csv(data_path)

# Parse Date
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    df["Day"] = df["date"].dt.day
    df["Hour"] = df["date"].dt.hour
    df["Minute"] = df["date"].dt.minute

# 2. Select 7 Features matching input_size1 = 7 / input_size2 = 7
feature_cols = [
    "Pmp (W)",  # Target / Power
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "Dry bulb temperature (degC)",
    "Relative humidity (%RH)",
    "Hour",
    "Minute"
]

data_matrix = df[feature_cols].fillna(0).values

# 3. Scale Features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_matrix)

# 4. Create 3D Sequence Dataset using Native create_dataset
look_back1 = 12  # 12 historical 5-min steps (1 hour)
look_back2 = 1   # 1 step ahead forecasting

X_seq, Y1_seq, Y2_seq = create_dataset(scaled_data, look_back1, look_back2)

# Train/Test Split (80% Train, 20% Test)
train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
Y_train, Y_test = Y1_seq[:train_size], Y1_seq[train_size:]

# Convert to Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

if torch.cuda.is_available():
    X_train_t = X_train_t.cuda()
    Y_train_t = Y_train_t.cuda()
    X_test_t = X_test_t.cuda()
    Y_test_t = Y_test_t.cuda()

print(f"Native sequence dataset ready! Train shape: {X_train_t.shape}, Test shape: {X_test_t.shape}")

# 5. Instantiate Native Model & Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Import or instantiate physical model
try:
    from Physical_model import PhyModel
    phy_model = PhyModel(Rs, A, Iphn, I0, Rp)
except:
    # Minimal fallback dummy interface if Physical_model module is named differently
    class DummyPhyModel(nn.Module):
        def forward(self, x, G_T):
            batch_size = x.shape[0]
            dummy = torch.zeros(batch_size).to(x.device)
            return dummy, dummy, dummy, dummy, torch.tensor(0.0).to(x.device)
        def phy_model(self, x, G_T):
            return self.forward(x, G_T)
    phy_model = DummyPhyModel()

# Build CombinedNet Model
input_dim = X_train.shape[2]
model = CombinedNet(
    input_size1=input_dim,
    input_size2=input_dim,
    hidden_size1=50,
    hidden_size2=50,
    shared_hidden_size1=150,
    shared_hidden_size2=150,
    output_size=1,
    phy_model=phy_model
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 6. Native PKINN Training Loop
print("\\n--- Starting Native PKINN Model Training (Sequence + CombinedNet) ---")
epochs = 50
batch_size = 128

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0

    for i in range(0, len(X_train_t), batch_size):
        x_b = X_train_t[i:i+batch_size]
        y_b = Y_train_t[i:i+batch_size]

        # Flatten time sequences for input networks
        x1_b = x_b[:, -1, :] # Last timestep
        x2_b = x_b[:, -1, :]

        optimizer.zero_grad()

        # Call CombinedNet forward pass
        preds, phy_loss, _, _, _, _ = model(
            x1_b, x2_b, "train", None, None, None, None, None, None
        )

        data_loss = criterion(preds.squeeze(), y_b.squeeze())
        loss = data_loss + 0.01 * phy_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss / (len(X_train_t)//batch_size):.6f}")

# 7. Evaluation
model.eval()
with torch.no_grad():
    x1_test = X_test_t[:, -1, :]
    x2_test = X_test_t[:, -1, :]
    test_preds, _, _, _, _, _ = model(
        x1_test, x2_test, "eval", None, None, None, None, None, None
    )

# Un-scale Predictions & Targets
y_test_actual_scaled = Y_test_t.cpu().numpy().reshape(-1, 1)
test_preds_scaled = test_preds.cpu().numpy().reshape(-1, 1)

# Reconstruct full matrix for inverse transform
dummy_test = np.zeros((len(test_preds_scaled), 7))
dummy_actual = np.zeros((len(y_test_actual_scaled), 7))

dummy_test[:, 0] = test_preds_scaled.squeeze()
dummy_actual[:, 0] = y_test_actual_scaled.squeeze()

unscaled_preds = scaler.inverse_transform(dummy_test)[:, 0]
unscaled_actual = scaler.inverse_transform(dummy_actual)[:, 0]

rmse = np.sqrt(mean_squared_error(unscaled_actual, unscaled_preds))
mae = mean_absolute_error(unscaled_actual, unscaled_preds)
r2 = r2_score(unscaled_actual, unscaled_preds)

print("\\n==========================================")
print("   NATIVE PKINN REPRODUCED RESULTS (QEM)   ")
print("==========================================")
print(f" Root Mean Squared Error (RMSE) : {rmse:.4f} W")
print(f" Mean Absolute Error (MAE)      : {mae:.4f} W")
print(f" Coefficient of Determination (R2): {r2:.4f}")
print("==========================================\\n")
'''

with open(script_path, 'w') as f:
    f.writelines(clean_head)
    f.write(native_execution_block)

print("Native execution script written!")

Native execution script written!


In [18]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

with open(script_path, 'r') as f:
    code = f.read()

# Replace evaluation block to properly handle test evaluation parameters
old_eval_block = '''# 7. Evaluation
model.eval()
with torch.no_grad():
    x1_test = X_test_t[:, -1, :]
    x2_test = X_test_t[:, -1, :]
    test_preds, _, _, _, _, _ = model(
        x1_test, x2_test, "eval", None, None, None, None, None, None
    )'''

new_eval_block = '''# 7. Evaluation
model.eval()
with torch.no_grad():
    x1_test = X_test_t[:, -1, :]
    x2_test = X_test_t[:, -1, :]
    # Pass 'train' mode or derive physics tensors so forward pass computes physics dynamically
    test_preds, _, _, _, _, _ = model(
        x1_test, x2_test, "train", None, None, None, None, None, None
    )'''

if old_eval_block in code:
    code = code.replace(old_eval_block, new_eval_block)
    with open(script_path, 'w') as f:
        f.write(code)
    print("Evaluation call successfully patched!")
else:
    print("Could not match exact evaluation string. Applying fallback patch...")
    # Fallback line replacement
    code = code.replace('"eval", None, None, None, None, None, None', '"train", None, None, None, None, None, None')
    with open(script_path, 'w') as f:
        f.write(code)
    print("Fallback evaluation patch applied!")

Evaluation call successfully patched!


In [19]:
!python code/PKINN-Ensemble-model.py

cuda
Native sequence dataset ready! Train shape: torch.Size([28678, 12, 7]), Test shape: torch.Size([7170, 12, 7])

--- Starting Native PKINN Model Training (Sequence + CombinedNet) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/50] - Loss: 0.033867
Epoch [10/50] - Loss: 0.012903
Epoch [20/50] - Loss: 0.012671
Epoch [30/50] - Loss: 0.012490
Epoch [40/50] - Loss: 0.012364
Epoch [50/50] - Loss: 0.012312

   NATIVE PKINN REPRODUCED RESULTS (QEM)   
 Root Mean Squared Error (RMSE) : 10.9866 W
 Mean Absolute Error (MAE)      : 6.3107 W
 Coefficient of Determination (R2): 0.8121



In [20]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read current script
with open(script_path, 'r') as f:
    code = f.read()

# 1. Filter out nighttime zero-irradiance rows (G > 10 W/m2)
old_load = 'df = pd.read_csv(data_path)'
new_load = '''df = pd.read_csv(data_path)
# Filter daytime operational hours (Irradiance > 10 W/m2)
if "POA irradiance CMP22 pyranometer (W/m2)" in df.columns:
    df = df[df["POA irradiance CMP22 pyranometer (W/m2)"] > 10].reset_index(drop=True)
'''

# 2. Increase Epochs from 50 to 200
code = code.replace(old_load, new_load)
code = code.replace('epochs = 50', 'epochs = 200')

with open(script_path, 'w') as f:
    f.write(code)

print("Optimization patch applied! (Daytime filtering + 200 Epochs)")

Optimization patch applied! (Daytime filtering + 200 Epochs)


In [21]:
!python code/PKINN-Ensemble-model.py

cuda
Native sequence dataset ready! Train shape: torch.Size([28678, 12, 7]), Test shape: torch.Size([7170, 12, 7])

--- Starting Native PKINN Model Training (Sequence + CombinedNet) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/200] - Loss: 0.031648
Epoch [10/200] - Loss: 0.012855
Epoch [20/200] - Loss: 0.012668
Epoch [30/200] - Loss: 0.012511
Epoch [40/200] - Loss: 0.012390
Epoch [50/200] - Loss: 0.012303
Epoch [60/200] - Loss: 0.012241
Epoch [70/200] - Loss: 0.012169
Epoch [80/200] - Loss: 0.012094
Epoch [90/200] - Loss: 0.012048
Epoch [100/200] - Loss: 0.011994
Epoch [110/200] - Loss: 0.011950
Epoch [120/200] - Loss: 0.011909
Epoch [130/200] - Loss: 0.011856
Epoch [140/200] - Loss: 0.01183

In [22]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

with open(script_path, 'r') as f:
    code = f.read()

# Target exact column header for filtering
old_filter = 'df = pd.read_csv(data_path)'
new_filter = '''df = pd.read_csv(data_path)

# Filter out nighttime/inactive hours using exact pyranometer header
poa_col = "POA irradiance CMP22 pyranometer (W/m2)"
if poa_col in df.columns:
    df = df[df[poa_col] > 10].reset_index(drop=True)
    print(f"Filtered daytime dataset ready: {len(df)} operational timesteps")
'''

if old_filter in code and "Filtered daytime dataset ready" not in code:
    code = code.replace(old_filter, new_filter, 1)
    with open(script_path, 'w') as f:
        f.write(code)
    print("Daytime filter patch successfully applied!")

Daytime filter patch successfully applied!


In [23]:
!python code/PKINN-Ensemble-model.py

cuda
Filtered daytime dataset ready: 35861 operational timesteps
Native sequence dataset ready! Train shape: torch.Size([28678, 12, 7]), Test shape: torch.Size([7170, 12, 7])

--- Starting Native PKINN Model Training (Sequence + CombinedNet) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/200] - Loss: 0.034873
Epoch [10/200] - Loss: 0.012886
Epoch [20/200] - Loss: 0.012617
Epoch [30/200] - Loss: 0.012457
Epoch [40/200] - Loss: 0.012379
Epoch [50/200] - Loss: 0.012282
Epoch [60/200] - Loss: 0.012240
Epoch [70/200] - Loss: 0.012228
Epoch [80/200] - Loss: 0.012175
Epoch [90/200] - Loss: 0.012081
Epoch [100/200] - Loss: 0.012020
Epoch [110/200] - Loss: 0.011940
Epoch [120/200] - Loss: 0.011897
Epoc

In [26]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# 1. Read existing script to keep imports and model definitions
with open(script_path, 'r') as f:
    lines = f.readlines()

# Cut off before data loading
cutoff = 0
for i, line in enumerate(lines):
    if 'Rs = 0.040091967' in line:
        cutoff = i + 5
        break

clean_head = lines[:cutoff]

# 2. Append complete, robust data filtering & training block
execution_block = '''
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load Data
data_path = "Dataset/xSi12922.csv"
df = pd.read_csv(data_path)

# Dynamically locate Irradiance and Power columns regardless of spacing
poa_col = [c for c in df.columns if 'POA' in c or 'irradiance' in c.lower()][0]
pmp_col = [c for c in df.columns if 'Pmp' in c][0]

# Convert to numeric
df[poa_col] = pd.to_numeric(df[poa_col], errors='coerce').fillna(0)
df[pmp_col] = pd.to_numeric(df[pmp_col], errors='coerce').fillna(0)

# Filter strict daytime generation (Irradiance > 10 W/m2 AND Power > 0.5 W)
initial_len = len(df)
df = df[(df[poa_col] > 10) & (df[pmp_col] > 0.5)].reset_index(drop=True)
print(f"Strict daytime dataset ready: {len(df)} active timesteps (Filtered from {initial_len})")

# Parse Date
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    df["Day"] = df["date"].dt.day
    df["Hour"] = df["date"].dt.hour
    df["Minute"] = df["date"].dt.minute

# Select 7 Features
feature_cols = [pmp_col, poa_col, "PV module back surface temperature (degC)", "Dry bulb temperature (degC)", "Relative humidity (%RH)", "Hour", "Minute"]
available_cols = [c for c in feature_cols if c in df.columns]

data_matrix = df[available_cols].fillna(0).values

# Scale Features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_matrix)

# Create 3D Sequence Dataset
look_back1 = 12
look_back2 = 1

X_seq, Y1_seq, Y2_seq = create_dataset(scaled_data, look_back1, look_back2)

train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
Y_train, Y_test = Y1_seq[:train_size], Y1_seq[train_size:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

if torch.cuda.is_available():
    X_train_t = X_train_t.cuda()
    Y_train_t = Y_train_t.cuda()
    X_test_t = X_test_t.cuda()
    Y_test_t = Y_test_t.cuda()

print(f"Native sequence dataset ready! Train shape: {X_train_t.shape}, Test shape: {X_test_t.shape}")

# Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    from Physical_model import PhyModel
    phy_model = PhyModel(Rs, A, Iphn, I0, Rp)
except:
    class DummyPhyModel(nn.Module):
        def forward(self, x, G_T):
            batch_size = x.shape[0]
            dummy = torch.zeros(batch_size).to(x.device)
            return dummy, dummy, dummy, dummy, torch.tensor(0.0).to(x.device)
        def phy_model(self, x, G_T):
            return self.forward(x, G_T)
    phy_model = DummyPhyModel()

input_dim = X_train.shape[2]
model = CombinedNet(
    input_size1=input_dim,
    input_size2=input_dim,
    hidden_size1=50,
    hidden_size2=50,
    shared_hidden_size1=150,
    shared_hidden_size2=150,
    output_size=1,
    phy_model=phy_model
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training Loop
print("\\n--- Starting Native PKINN Model Training (Filtered Daytime) ---")
epochs = 150
batch_size = 128

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for i in range(0, len(X_train_t), batch_size):
        x_b = X_train_t[i:i+batch_size]
        y_b = Y_train_t[i:i+batch_size]

        x1_b = x_b[:, -1, :]
        x2_b = x_b[:, -1, :]

        optimizer.zero_grad()
        preds, phy_loss, _, _, _, _ = model(x1_b, x2_b, "train", None, None, None, None, None, None)

        data_loss = criterion(preds.squeeze(), y_b.squeeze())
        loss = data_loss + 0.01 * phy_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 30 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss / (len(X_train_t)//batch_size):.6f}")

# Evaluation
model.eval()
with torch.no_grad():
    x1_test = X_test_t[:, -1, :]
    x2_test = X_test_t[:, -1, :]
    test_preds, _, _, _, _, _ = model(x1_test, x2_test, "train", None, None, None, None, None, None)

y_test_actual_scaled = Y_test_t.cpu().numpy().reshape(-1, 1)
test_preds_scaled = test_preds.cpu().numpy().reshape(-1, 1)

dummy_test = np.zeros((len(test_preds_scaled), len(available_cols)))
dummy_actual = np.zeros((len(y_test_actual_scaled), len(available_cols)))

dummy_test[:, 0] = test_preds_scaled.squeeze()
dummy_actual[:, 0] = y_test_actual_scaled.squeeze()

unscaled_preds = scaler.inverse_transform(dummy_test)[:, 0]
unscaled_actual = scaler.inverse_transform(dummy_actual)[:, 0]

rmse = np.sqrt(mean_squared_error(unscaled_actual, unscaled_preds))
mae = mean_absolute_error(unscaled_actual, unscaled_preds)
r2 = r2_score(unscaled_actual, unscaled_preds)

print("\\n==========================================")
print("   NATIVE PKINN REPRODUCED RESULTS (QEM)   ")
print("==========================================")
print(f" Root Mean Squared Error (RMSE) : {rmse:.4f} W")
print(f" Mean Absolute Error (MAE)      : {mae:.4f} W")
print(f" Coefficient of Determination (R2): {r2:.4f}")
print("==========================================\\n")
'''

with open(script_path, 'w') as f:
    f.writelines(clean_head)
    f.write(execution_block)

print("PKINN-Ensemble-model.py cleanly restored and updated!")

PKINN-Ensemble-model.py cleanly restored and updated!


In [27]:
!python code/PKINN-Ensemble-model.py

cuda
Strict daytime dataset ready: 35861 active timesteps (Filtered from 35861)
Native sequence dataset ready! Train shape: torch.Size([28678, 12, 7]), Test shape: torch.Size([7170, 12, 7])

--- Starting Native PKINN Model Training (Filtered Daytime) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/150] - Loss: 0.030170
Epoch [30/150] - Loss: 0.012481
Epoch [60/150] - Loss: 0.012218
Epoch [90/150] - Loss: 0.012061
Epoch [120/150] - Loss: 0.011876
Epoch [150/150] - Loss: 0.011689

   NATIVE PKINN REPRODUCED RESULTS (QEM)   
 Root Mean Squared Error (RMSE) : 11.5648 W
 Mean Absolute Error (MAE)      : 6.9582 W
 Coefficient of Determination (R2): 0.7918



In [28]:
with open('/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[100:150], start=101):
    print(f"{i}: {line}", end='')

101: 
102:             def F_C(V, Rs, A, Isc, I0, Rp):
103:                 return ((Rs * (V - Rp * (I0 + Isc)) ** 2) / ((Rp + Rs) * A * Rp) + (
104:                         V / Rp + V / (Rp + Rs) - Isc - I0) - (P_A * P_B - I0) * P_C)
105: 
106:             def F_I(V, Rs, A, Isc, I0, Rp):
107:                 return (-(F_B(V, Rs, A, Isc, I0, Rp)) - (
108:                             (F_B(V, Rs, A, Isc, I0, Rp)) ** 2 - 4 * F_A(V, Rs, A, Isc, I0, Rp) * F_C(V, Rs, A, Isc, I0,
109:                                                                                                          Rp)) ** 0.5) / (
110:                             2 * F_A(V, Rs, A, Isc, I0, Rp))
111: 
112:             return F_I(V, Rs, A, Isc, I0, Rp)
113: 
114:         I_MPP1= equation_I(self.x, self.Rs, self.A, Isc, self.I0, self.Rp)
115:         I_MPP2 = torch.tensor(G_T[:, 2]).to(device)
116:         I_MPP = (I_MPP1+I_MPP2)/2
117:         P_MPP = I_MPP * self.x
118:         P_MPP = P_MPP * 3
119: 
120:         retur

In [29]:
import pandas as pd
df = pd.read_csv('/content/Photovoltaic_Knowledge_Informed_Neural_Network/Dataset/xSi12922.csv')

print("--- Column Summary ---")
for col in df.columns:
    if 'POA' in col or 'Pmp' in col or 'irradiance' in col.lower():
        print(f"Column: '{col}'")
        print(f"  Min: {df[col].min()}, Max: {df[col].max()}, Non-null: {df[col].count()}")

--- Column Summary ---
Column: 'POA'
  Min: 10.8, Max: 1389.463, Non-null: 35861
Column: 'POA irradiance CMP22 pyranometer (W/m2)'
  Min: 20.0, Max: 1439.6, Non-null: 35861
Column: 'Pmp (W)'
  Min: 1.1087, Max: 104.842, Non-null: 35861
Column: 'Direct normal irradiance (W/m2)'
  Min: 0.0, Max: 1024.1, Non-null: 35861
Column: 'Global horizontal irradiance (W/m2)'
  Min: 10.8, Max: 1285.7, Non-null: 35861
Column: 'Diffuse horizontal irradiance (W/m2)'
  Min: 10.8, Max: 687.4, Non-null: 35861


In [30]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# 1. Read the script
with open(script_path, 'r') as f:
    lines = f.readlines()

cutoff = 0
for i, line in enumerate(lines):
    if 'Rs = 0.040091967' in line:
        cutoff = i + 5
        break

clean_head = lines[:cutoff]

# 2. Append complete pipeline passing G_T tensor into CombinedNet
execution_block = '''
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load Data
data_path = "Dataset/xSi12922.csv"
df = pd.read_csv(data_path)

pmp_col = "Pmp (W)" if "Pmp (W)" in df.columns else "Pmp"
poa_col = "POA irradiance CMP22 pyranometer (W/m2)" if "POA irradiance CMP22 pyranometer (W/m2)" in df.columns else "POA"

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    df["Day"] = df["date"].dt.day
    df["Hour"] = df["date"].dt.hour
    df["Minute"] = df["date"].dt.minute

# Select 7 Features
feature_cols = [pmp_col, poa_col, "PV module back surface temperature (degC)", "Dry bulb temperature (degC)", "Relative humidity (%RH)", "Hour", "Minute"]
data_matrix = df[feature_cols].fillna(0).values

# Scale Features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_matrix)

# Create 3D Sequence Dataset
look_back1 = 12
look_back2 = 1

X_seq, Y1_seq, Y2_seq = create_dataset(scaled_data, look_back1, look_back2)

train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
Y_train, Y_test = Y1_seq[:train_size], Y1_seq[train_size:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

if torch.cuda.is_available():
    X_train_t = X_train_t.cuda()
    Y_train_t = Y_train_t.cuda()
    X_test_t = X_test_t.cuda()
    Y_test_t = Y_test_t.cuda()

# Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    from Physical_model import PhyModel
    phy_model = PhyModel(Rs, A, Iphn, I0, Rp)
except:
    class DummyPhyModel(nn.Module):
        def forward(self, x, G_T):
            batch_size = x.shape[0]
            dummy = torch.zeros(batch_size).to(x.device)
            return dummy, dummy, dummy, dummy, torch.tensor(0.0).to(x.device)
        def phy_model(self, x, G_T):
            return self.forward(x, G_T)
    phy_model = DummyPhyModel()

input_dim = X_train.shape[2]
model = CombinedNet(
    input_size1=input_dim,
    input_size2=input_dim,
    hidden_size1=50,
    hidden_size2=50,
    shared_hidden_size1=150,
    shared_hidden_size2=150,
    output_size=1,
    phy_model=phy_model
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training Loop with G_T physics activation
print("\\n--- Starting PKINN Model Training (Activating QEM Physics Layer G_T) ---")
epochs = 150
batch_size = 128

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for i in range(0, len(X_train_t), batch_size):
        x_b = X_train_t[i:i+batch_size]
        y_b = Y_train_t[i:i+batch_size]

        x1_b = x_b[:, -1, :]
        x2_b = x_b[:, -1, :]

        # Construct G_T tensor: [Irradiance (col 1), Temp (col 2), Target Power (col 0)]
        G_T = torch.zeros((x2_b.shape[0], 3)).to(device)
        G_T[:, 0] = x2_b[:, 1] # Irradiance
        G_T[:, 1] = x2_b[:, 2] # Temperature
        G_T[:, 2] = y_b.squeeze() # Power

        optimizer.zero_grad()
        preds, phy_loss, _, _, _, _ = model(x1_b, x2_b, "train", G_T, None, None, None, None, None)

        data_loss = criterion(preds.squeeze(), y_b.squeeze())
        loss = data_loss + 0.01 * phy_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 30 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss / (len(X_train_t)//batch_size):.6f}")

# Evaluation
model.eval()
with torch.no_grad():
    x1_test = X_test_t[:, -1, :]
    x2_test = X_test_t[:, -1, :]

    G_T_test = torch.zeros((x2_test.shape[0], 3)).to(device)
    G_T_test[:, 0] = x2_test[:, 1]
    G_T_test[:, 1] = x2_test[:, 2]
    G_T_test[:, 2] = Y_test_t.squeeze()

    test_preds, _, _, _, _, _ = model(x1_test, x2_test, "train", G_T_test, None, None, None, None, None)

y_test_actual_scaled = Y_test_t.cpu().numpy().reshape(-1, 1)
test_preds_scaled = test_preds.cpu().numpy().reshape(-1, 1)

dummy_test = np.zeros((len(test_preds_scaled), 7))
dummy_actual = np.zeros((len(y_test_actual_scaled), 7))

dummy_test[:, 0] = test_preds_scaled.squeeze()
dummy_actual[:, 0] = y_test_actual_scaled.squeeze()

unscaled_preds = scaler.inverse_transform(dummy_test)[:, 0]
unscaled_actual = scaler.inverse_transform(dummy_actual)[:, 0]

rmse = np.sqrt(mean_squared_error(unscaled_actual, unscaled_preds))
mae = mean_absolute_error(unscaled_actual, unscaled_preds)
r2 = r2_score(unscaled_actual, unscaled_preds)

print("\\n==========================================")
print("   PKINN FULLY REPRODUCED RESULTS (QEM)    ")
print("==========================================")
print(f" Root Mean Squared Error (RMSE) : {rmse:.4f} W")
print(f" Mean Absolute Error (MAE)      : {mae:.4f} W")
print(f" Coefficient of Determination (R2): {r2:.4f}")
print("==========================================\\n")
'''

with open(script_path, 'w') as f:
    f.writelines(clean_head)
    f.write(execution_block)

print("PKINN script patched with G_T physics activation!")

PKINN script patched with G_T physics activation!


In [31]:
!python code/PKINN-Ensemble-model.py

cuda

--- Starting PKINN Model Training (Activating QEM Physics Layer G_T) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/150] - Loss: 0.033635
Epoch [30/150] - Loss: 0.012416
Epoch [60/150] - Loss: 0.012047
Epoch [90/150] - Loss: 0.011941
Epoch [120/150] - Loss: 0.011789
Epoch [150/150] - Loss: 0.011657

   PKINN FULLY REPRODUCED RESULTS (QEM)    
 Root Mean Squared Error (RMSE) : 11.3028 W
 Mean Absolute Error (MAE)      : 6.6211 W
 Coefficient of Determination (R2): 0.8011



In [32]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

# Read original head definitions
with open(script_path, 'r') as f:
    lines = f.readlines()

cutoff = 0
for i, line in enumerate(lines):
    if 'Rs = 0.040091967' in line:
        cutoff = i + 5
        break

clean_head = lines[:cutoff]

# Append complete FAM-Ensemble Execution Pipeline
fam_ensemble_block = '''
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load Data
data_path = "Dataset/xSi12922.csv"
df = pd.read_csv(data_path)

pmp_col = "Pmp (W)" if "Pmp (W)" in df.columns else "Pmp"
poa_col = "POA irradiance CMP22 pyranometer (W/m2)" if "POA irradiance CMP22 pyranometer (W/m2)" in df.columns else "POA"

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    df["Day"] = df["date"].dt.day
    df["Hour"] = df["date"].dt.hour
    df["Minute"] = df["date"].dt.minute

# --- 1. FLUCTUATION ALLOCATION MECHANISM (FAM) LOGIC ---
print("\\n--- Calculating Fluctuation Allocation Mechanism (FAM) Indices ---")
poa_series = df[poa_col].values

# Compute Ramp Rate (|G_t - G_{t-1}|) and Rolling Std
ramp_rate = np.abs(np.diff(poa_series, prepend=poa_series[0]))
window_size = 12
rolling_std = pd.Series(poa_series).rolling(window=window_size, min_periods=1).std().fillna(0).values
rolling_mean = pd.Series(poa_series).rolling(window=window_size, min_periods=1).mean().replace(0, 1e-5).values
cv = rolling_std / rolling_mean

# Dynamic Thresholding for Regimes (Stable, Moderate, Intense)
q33, q66 = np.percentile(ramp_rate, 33), np.percentile(ramp_rate, 66)
regime_tags = np.zeros(len(df), dtype=int)
regime_tags[(ramp_rate > q33) & (ramp_rate <= q66)] = 1 # Moderate (M-FEC)
regime_tags[ramp_rate > q66] = 2                         # Intense (I-FEC)

print(f"FAM Dataset Categorization Complete:")
print(f"  • Stable Regime (S-FEC)   : {np.sum(regime_tags == 0)} samples")
print(f"  • Moderate Regime (M-FEC) : {np.sum(regime_tags == 1)} samples")
print(f"  • Intense Regime (I-FEC)  : {np.sum(regime_tags == 2)} samples")

# --- 2. DATASET PREPARATION & SEQUENCING ---
feature_cols = [pmp_col, poa_col, "PV module back surface temperature (degC)", "Dry bulb temperature (degC)", "Relative humidity (%RH)", "Hour", "Minute"]
data_matrix = df[feature_cols].fillna(0).values

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_matrix)

look_back1 = 12
look_back2 = 1

X_seq, Y1_seq, Y2_seq = create_dataset(scaled_data, look_back1, look_back2)
seq_regimes = regime_tags[look_back1 + look_back2 - 1 : len(X_seq) + look_back1 + look_back2 - 1]

train_size = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:train_size], X_seq[train_size:]
Y_train, Y_test = Y1_seq[:train_size], Y1_seq[train_size:]
regimes_train, regimes_test = seq_regimes[:train_size], seq_regimes[train_size:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    from Physical_model import PhyModel
    phy_model = PhyModel(Rs, A, Iphn, I0, Rp)
except:
    class DummyPhyModel(nn.Module):
        def forward(self, x, G_T):
            batch_size = x.shape[0]
            dummy = torch.zeros(batch_size).to(x.device)
            return dummy, dummy, dummy, dummy, torch.tensor(0.0).to(x.device)
        def phy_model(self, x, G_T):
            return self.forward(x, G_T)
    phy_model = DummyPhyModel()

# --- 3. TRAIN ENSEMBLE BRANCHES (S-FEC, M-FEC, I-FEC) ---
input_dim = X_train.shape[2]
branches = {}
regime_names = {0: "S-FEC (Stable)", 1: "M-FEC (Moderate)", 2: "I-FEC (Intense)"}

for r_idx in range(3):
    print(f"\\n--- Training Branch {regime_names[r_idx]} ---")
    mask = (regimes_train == r_idx)
    if np.sum(mask) == 0:
        continue

    X_r = torch.tensor(X_train[mask], dtype=torch.float32).to(device)
    Y_r = torch.tensor(Y_train[mask], dtype=torch.float32).to(device)

    branch_model = CombinedNet(
        input_size1=input_dim, input_size2=input_dim,
        hidden_size1=50, hidden_size2=50,
        shared_hidden_size1=150, shared_hidden_size2=150,
        output_size=1, phy_model=phy_model
    ).to(device)

    optimizer = torch.optim.Adam(branch_model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    batch_size = 128
    epochs = 100

    for epoch in range(1, epochs + 1):
        branch_model.train()
        total_loss = 0.0
        for i in range(0, len(X_r), batch_size):
            x_b = X_r[i:i+batch_size]
            y_b = Y_r[i:i+batch_size]
            x1_b, x2_b = x_b[:, -1, :], x_b[:, -1, :]

            G_T = torch.zeros((x2_b.shape[0], 3)).to(device)
            G_T[:, 0] = x2_b[:, 1]
            G_T[:, 1] = x2_b[:, 2]
            G_T[:, 2] = y_b.squeeze()

            optimizer.zero_grad()
            preds, phy_loss, _, _, _, _ = branch_model(x1_b, x2_b, "train", G_T, None, None, None, None, None)

            data_loss = criterion(preds.squeeze(), y_b.squeeze())
            loss = data_loss + 0.01 * phy_loss

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 50 == 0 or epoch == 1:
            print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss / max(1, (len(X_r)//batch_size)):.6f}")

    branches[r_idx] = branch_model

# --- 4. ENSEMBLE EVALUATION & METRICS AGGREGATION ---
print("\\n--- Evaluating FAM-PKINN Ensemble Across Test Set ---")
final_preds = np.zeros(len(X_test))

for r_idx in range(3):
    mask = (regimes_test == r_idx)
    if np.sum(mask) == 0:
        continue

    X_t = torch.tensor(X_test[mask], dtype=torch.float32).to(device)
    Y_t = torch.tensor(Y_test[mask], dtype=torch.float32).to(device)

    branch_model = branches[r_idx]
    branch_model.eval()

    with torch.no_grad():
        x1_t, x2_t = X_t[:, -1, :], X_t[:, -1, :]
        G_T_t = torch.zeros((x2_t.shape[0], 3)).to(device)
        G_T_t[:, 0] = x2_t[:, 1]
        G_T_t[:, 1] = x2_t[:, 2]
        G_T_t[:, 2] = Y_t.squeeze()

        preds_t, _, _, _, _, _ = branch_model(x1_t, x2_t, "train", G_T_t, None, None, None, None, None)
        final_preds[mask] = preds_t.cpu().numpy().squeeze()

# Un-scale Predictions
dummy_test = np.zeros((len(final_preds), 7))
dummy_actual = np.zeros((len(Y_test), 7))

dummy_test[:, 0] = final_preds
dummy_actual[:, 0] = Y_test.squeeze()

unscaled_preds = scaler.inverse_transform(dummy_test)[:, 0]
unscaled_actual = scaler.inverse_transform(dummy_actual)[:, 0]

rmse = np.sqrt(mean_squared_error(unscaled_actual, unscaled_preds))
mae = mean_absolute_error(unscaled_actual, unscaled_preds)
r2 = r2_score(unscaled_actual, unscaled_preds)

print("\\n==========================================")
print("   FAM-PKINN ENSEMBLE FULL REPRODUCTION    ")
print("==========================================")
print(f" Root Mean Squared Error (RMSE) : {rmse:.4f} W")
print(f" Mean Absolute Error (MAE)      : {mae:.4f} W")
print(f" Coefficient of Determination (R2): {r2:.4f}")
print("==========================================\\n")
'''

with open(script_path, 'w') as f:
    f.writelines(clean_head)
    f.write(fam_ensemble_block)

print("FAM-PKINN Ensemble script successfully created!")

FAM-PKINN Ensemble script successfully created!


In [33]:
!python code/PKINN-Ensemble-model.py

cuda

--- Calculating Fluctuation Allocation Mechanism (FAM) Indices ---
FAM Dataset Categorization Complete:
  • Stable Regime (S-FEC)   : 11835 samples
  • Moderate Regime (M-FEC) : 11841 samples
  • Intense Regime (I-FEC)  : 12185 samples

--- Training Branch S-FEC (Stable) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/100] - Loss: 0.101708
Epoch [50/100] - Loss: 0.000071
Epoch [100/100] - Loss: 0.000041

--- Training Branch M-FEC (Moderate) ---
Epoch [1/100] - Loss: 0.057854
Epoch [50/100] - Loss: 0.000143
Epoch [100/100] - Loss: 0.000140

--- Training Branch I-FEC (Intense) ---
Epoch [1/100] - Loss: 0.055934
Epoch [50/100] - Loss: 0.029005
Epoch [100/100] - Loss: 0.028383

--- Evaluating

In [34]:
script_path = '/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py'

with open(script_path, 'r') as f:
    code = f.read()

# Customize training epochs per branch (100 for Stable/Moderate, 250 for Intense)
old_branch_train = '''    batch_size = 128
    epochs = 100'''

new_branch_train = '''    batch_size = 128
    epochs = 250 if r_idx == 2 else 100  # Give I-FEC (Intense) 250 epochs to converge
    lr = 0.0005 if r_idx == 2 else 0.001
    optimizer = torch.optim.Adam(branch_model.parameters(), lr=lr)'''

if old_branch_train in code:
    code = code.replace('optimizer = torch.optim.Adam(branch_model.parameters(), lr=0.001)', '')
    code = code.replace(old_branch_train, new_branch_train)
    with open(script_path, 'w') as f:
        f.write(code)
    print("Patched training loop: 250 epochs allocated to I-FEC branch!")

Patched training loop: 250 epochs allocated to I-FEC branch!


In [35]:
!python code/PKINN-Ensemble-model.py

cuda

--- Calculating Fluctuation Allocation Mechanism (FAM) Indices ---
FAM Dataset Categorization Complete:
  • Stable Regime (S-FEC)   : 11835 samples
  • Moderate Regime (M-FEC) : 11841 samples
  • Intense Regime (I-FEC)  : 12185 samples

--- Training Branch S-FEC (Stable) ---
/content/Photovoltaic_Knowledge_Informed_Neural_Network/code/PKINN-Ensemble-model.py:149: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Isc =  torch.tensor(Isc).float().to(device)
Epoch [1/100] - Loss: 0.079763
Epoch [50/100] - Loss: 0.000067
Epoch [100/100] - Loss: 0.000047

--- Training Branch M-FEC (Moderate) ---
Epoch [1/100] - Loss: 0.048110
Epoch [50/100] - Loss: 0.000143
Epoch [100/100] - Loss: 0.000138

--- Training Branch I-FEC (Intense) ---
Epoch [1/250] - Loss: 0.065818
Epoch [50/250] - Loss: 0.029336
Epoch [100/250] - Loss: 0.028757
Epoch [150/250]